# Customer Data Processing

### Imports

In [0]:
from pyspark.sql.functions import *

### Schema setup for Bronze, Silver and Gold layers

In [0]:
bronze_schema = 'bronze'
silver_schema = 'silver'
gold_schema = 'gold'

### Defining widgets for runtime configuration

#### Syntax: dbutils.widgets.text(internal handle, pre-filled value, display text)

#### /*.csv = 'read every csv file in this folder'

In [0]:
dbutils.widgets.text('catalog', 'fmcg', 'Catalog')
dbutils.widgets.text('data_source', 'customers', 'Data Source')

catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

print(catalog)
print(data_source)

base_path = f's3://databricks-data-engineering-project/{data_source}/*.csv'
print(base_path)

fmcg
customers
s3://databricks-data-engineering-project/customers/*.csv


### Read CSV from source

In [0]:
df = (
    spark.read.format('csv')
    .option('header', True)
    .option('inferSchema', True)
    .load(base_path)
    .withColumn('read_timestamp', current_timestamp())
    .select('*', '_metadata.file_name', '_metadata.file_size')
)
display(df.limit(10))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-10-23T12:38:37.654Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-10-23T12:38:37.654Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-10-23T12:38:37.654Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-10-23T12:38:37.654Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-10-23T12:38:37.654Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-10-23T12:38:37.654Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-10-23T12:38:37.654Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-10-23T12:38:37.654Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-10-23T12:38:37.654Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-10-23T12:38:37.654Z,customers.csv,1404


### Inspect the inferred schema

In [0]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



### Ingesting raw data into Bronze layer

In [0]:
(
    df.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')
)

### Silver Processing

In [0]:
df_bronze = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source};')
df_bronze.show(10)

+-----------+--------------------+---------+--------------------+-------------+---------+
|customer_id|       customer_name|     city|      read_timestamp|    file_name|file_size|
+-----------+--------------------+---------+--------------------+-------------+---------+
|     789201|      FitFuel Market|Bengaluru|2026-10-23 12:38:...|customers.csv|     1404|
|     789202|      FitFuel Market|Hyderabad|2026-10-23 12:38:...|customers.csv|     1404|
|     789203|      FitFuel Market|New Delhi|2026-10-23 12:38:...|customers.csv|     1404|
|     789301|Athlete's Choice ...|Bengaluru|2026-10-23 12:38:...|customers.csv|     1404|
|     789303|Athlete's Choice ...|New Delhi|2026-10-23 12:38:...|customers.csv|     1404|
|     789101|     Endurance Foods|Bengalore|2026-10-23 12:38:...|customers.csv|     1404|
|     789102|     Endurance Foods|Hyderabad|2026-10-23 12:38:...|customers.csv|     1404|
|     789103|     Endurance Foods|New Delhi|2026-10-23 12:38:...|customers.csv|     1404|
|     7891

### Check for duplicate customers

In [0]:
display(spark.sql(f"""
SELECT
    customer_id,
    COUNT(customer_id) AS duplicate_count
FROM
    {catalog}.{bronze_schema}.{data_source}
GROUP BY
    customer_id
HAVING
    COUNT(customer_id) > 1;
"""))

customer_id,duplicate_count
789321,2
789503,2
789522,2
789603,2


### Deduplicate on `customer_id` and report row counts

In [0]:
print(f'Rows before dedupe: {df_bronze.count()}')

df_silver = df_bronze.drop_duplicates(subset = ['customer_id'])
print(f'Rows after dedupe: {df_silver.count()}')

Rows before dedupe: 39
Rows after dedupe: 35


### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Check for whitespaces in `customer_name`

In [0]:
display(spark.sql(f"""
SELECT
    *
FROM
    {catalog}.{silver_schema}.{data_source}
WHERE
    LEN(customer_name) <> LEN(TRIM(customer_name));
"""))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789420,ZenAthlete foods,null,2026-10-23T12:38:49.621Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-10-23T12:38:49.621Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-10-23T12:38:49.621Z,customers.csv,1404
789521,PrimeFuel Nutrition,null,2026-10-23T12:38:49.621Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-10-23T12:38:49.621Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-10-23T12:38:49.621Z,customers.csv,1404


### Trim whitespaces

##### P.S: withColumn adds or replaces column

In [0]:
df_silver = df_silver.withColumn('customer_name', trim(col('customer_name')))

### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Verify trim applied

In [0]:
display(spark.sql(f"""
SELECT
    *
FROM
    {catalog}.{silver_schema}.{data_source}
WHERE
    LEN(customer_name) <> LEN(TRIM(customer_name));
"""))

customer_id,customer_name,city,read_timestamp,file_name,file_size


### Distinct cities in silver

In [0]:
display(spark.sql(f"""
SELECT
    DISTINCT(city)
FROM
    {catalog}.{silver_schema}.{data_source};
"""))

city
New Delhi
Hyderabad
Bengaluru
Hyderbad
NewDelhi
Bengaluruu
Bengalore
NewDheli
Hyderabadd
NewDelhee


### Clean and standardize city values

In [0]:
city_cleaning_map = {
    'Hyderbad'      : 'Hyderabad',
    'Hyderabadd'    : 'Hyderabad',

    'NewDelhi'      : 'New Delhi',
    'NewDheli'      : 'New Delhi',
    'NewDelhee'     : 'New Delhi',

    'Bengalore'     : 'Bengaluru',
    'Bengaluruu'    : 'Bengaluru'
}

allowed = ['Hyderabad', 'New Delhi', 'Bengaluru']

df_silver = df_silver.replace(city_cleaning_map, subset = ['city'])

df_silver = df_silver.withColumn(
    'city', 
    when(col('city').isin(allowed), col('city')).otherwise(None))

### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)

### Verify distinct cities after cleaning

In [0]:
display(spark.sql(f"""
SELECT
    DISTINCT(city)
FROM
    {catalog}.{silver_schema}.{data_source};
"""))

city
New Delhi
Hyderabad
Bengaluru
null



### Distinct customer names in silver

In [0]:
display(spark.sql(f"""
SELECT
    DISTINCT customer_name
FROM
    {catalog}.{silver_schema}.{data_source};
"""))

customer_name
EliteAthlete Nutrition
PowerSnack hub
Recovery Lane
GamePlan Foods
FitFuel Market
Athlete's Choice Store
ZenAthlete foods
HydroBoost Nutrition
ZenAthlete Foods
champion's Choice


### Clean and standardize customer names

In [0]:
df_silver = df_silver.withColumn(
    'customer_name',
    initcap(col('customer_name')))

### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)


### Verify customer names after cleaning

In [0]:
display(spark.sql(f"""
SELECT
    DISTINCT customer_name
FROM
    {catalog}.{silver_schema}.{data_source};
"""))

customer_name
Eliteathlete Nutrition
Powersnack Hub
Recovery Lane
Gameplan Foods
Fitfuel Market
Athlete's Choice Store
Zenathlete Foods
Hydroboost Nutrition
Champion's Choice
Peak Performance Store



### Review rows with `NULL` city

In [0]:
display(spark.sql(f"""
SELECT
    *
FROM
    {catalog}.{silver_schema}.{data_source}
WHERE
    city IS NULL;
"""))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789420,Zenathlete Foods,null,2026-10-23T12:38:49.621Z,customers.csv,1404
789403,Sprintx Nutrition,null,2026-10-23T12:38:49.621Z,customers.csv,1404
789521,Primefuel Nutrition,null,2026-10-23T12:38:49.621Z,customers.csv,1404
789603,Recovery Lane,null,2026-10-23T12:38:49.621Z,customers.csv,1404


### Inspect Customers with Missing City

In [0]:
customers_missing_city = ['Zenathlete Foods', 'Sprintx Nutrition', 'Primefuel Nutrition', 'Recovery Lane']

df_silver.filter(col('customer_name').isin(customers_missing_city)).show()

+-----------+-------------------+---------+--------------------+-------------+---------+
|customer_id|      customer_name|     city|      read_timestamp|    file_name|file_size|
+-----------+-------------------+---------+--------------------+-------------+---------+
|     789401|  Sprintx Nutrition|Bengaluru|2026-10-23 12:38:...|customers.csv|     1404|
|     789402|  Sprintx Nutrition|Hyderabad|2026-10-23 12:38:...|customers.csv|     1404|
|     789403|  Sprintx Nutrition|     NULL|2026-10-23 12:38:...|customers.csv|     1404|
|     789420|   Zenathlete Foods|     NULL|2026-10-23 12:38:...|customers.csv|     1404|
|     789421|   Zenathlete Foods|Hyderabad|2026-10-23 12:38:...|customers.csv|     1404|
|     789422|   Zenathlete Foods|New Delhi|2026-10-23 12:38:...|customers.csv|     1404|
|     789520|Primefuel Nutrition|Bengaluru|2026-10-23 12:38:...|customers.csv|     1404|
|     789521|Primefuel Nutrition|     NULL|2026-10-23 12:38:...|customers.csv|     1404|
|     789522|Primefue


### Customer-to-City fix map

In [0]:
customer_city_map = {
    # Zenathlete Foods
    789420 : 'Bengaluru',

    # Sprintx Nutrition
    789403 : 'New Delhi',

    # Primefuel Nutrition
    789521 : 'Hyderabad',

    # Recovery Lane
    789603 : 'Hyderabad'
}

### Build customer_city_fix dataframe

In [0]:
customer_city_fix = spark.createDataFrame(customer_city_map.items(), schema = ['customer_id', 'city_fix'])

customer_city_fix.show()

+-----------+---------+
|customer_id| city_fix|
+-----------+---------+
|     789420|Bengaluru|
|     789403|New Delhi|
|     789521|Hyderabad|
|     789603|Hyderabad|
+-----------+---------+




### Fill missing city from the fix map

In [0]:
df_silver = (
    df_silver.alias('silver')
    .join(customer_city_fix.alias('fix'), on = 'customer_id', how = 'left')
    .withColumn('city', coalesce(col('silver.city'), col('fix.city_fix')))
    .drop(col('fix.city_fix'))
)

display(df_silver)

customer_id,customer_name,city,read_timestamp,file_name,file_size
789622,Eliteathlete Nutrition,New Delhi,2026-10-23T12:38:49.621Z,customers.csv,1404
789321,Powersnack Hub,Hyderabad,2026-10-23T12:38:49.621Z,customers.csv,1404
789601,Recovery Lane,Bengaluru,2026-10-23T12:38:49.621Z,customers.csv,1404
789720,Gameplan Foods,Bengaluru,2026-10-23T12:38:49.621Z,customers.csv,1404
789201,Fitfuel Market,Bengaluru,2026-10-23T12:38:49.621Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-10-23T12:38:49.621Z,customers.csv,1404
789420,Zenathlete Foods,Bengaluru,2026-10-23T12:38:49.621Z,customers.csv,1404
789202,Fitfuel Market,Hyderabad,2026-10-23T12:38:49.621Z,customers.csv,1404
789122,Hydroboost Nutrition,New Delhi,2026-10-23T12:38:49.621Z,customers.csv,1404
789421,Zenathlete Foods,Hyderabad,2026-10-23T12:38:49.621Z,customers.csv,1404


### Write Transformed Data to Silver

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)


### Verify city fix applied

In [0]:
display(spark.sql(f"""
SELECT
    *
FROM
    {catalog}.{silver_schema}.{data_source}
WHERE
    customer_name IN 
    ('Zenathlete Foods', 'Sprintx Nutrition', 'Primefuel Nutrition', 'Recovery Lane')
"""))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789601,Recovery Lane,Bengaluru,2026-10-23T12:38:49.621Z,customers.csv,1404
789420,Zenathlete Foods,Bengaluru,2026-10-23T12:38:49.621Z,customers.csv,1404
789421,Zenathlete Foods,Hyderabad,2026-10-23T12:38:49.621Z,customers.csv,1404
789520,Primefuel Nutrition,Bengaluru,2026-10-23T12:38:49.621Z,customers.csv,1404
789403,Sprintx Nutrition,New Delhi,2026-10-23T12:38:49.621Z,customers.csv,1404
789521,Primefuel Nutrition,Hyderabad,2026-10-23T12:38:49.621Z,customers.csv,1404
789401,Sprintx Nutrition,Bengaluru,2026-10-23T12:38:49.621Z,customers.csv,1404
789402,Sprintx Nutrition,Hyderabad,2026-10-23T12:38:49.621Z,customers.csv,1404
789522,Primefuel Nutrition,New Delhi,2026-10-23T12:38:49.621Z,customers.csv,1404
789603,Recovery Lane,Hyderabad,2026-10-23T12:38:49.621Z,customers.csv,1404


### Cast `customer_id` to string

In [0]:
df_silver = df_silver.withColumn('customer_id', col('customer_id').cast('string'))

df_silver.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)




### Build customer table and add constant columns

In [0]:
df_silver = df_silver.withColumn('customer', concat_ws(' - ', col('customer_name'), col('city')))

df_silver = (
    df_silver
    .withColumn('market', lit('Inida'))
    .withColumn('platform', lit('Sports Bar'))
    .withColumn('channel', lit('Acquisition'))
)

### Write Transformed Data to Silver
> `overwriteSchema=True` is required when a column is added, dropped, renamed, or when there is a change in datatype. 

In [0]:
(
    df_silver.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .option('overwriteSchema', True)
    .saveAsTable(f'{catalog}.{silver_schema}.{data_source}')
)


### Gold Processing - Select final columns

In [0]:
df_gold = df_silver.select('customer_id', 'customer_name', 'city', 'customer', 'market', 'platform', 'channel')

### Writing staging dimension to gold schema

In [0]:
(
    df_gold.write
    .format('delta')
    .option('delta.enableChangeDataFeed', True)
    .mode('overwrite')
    .saveAsTable(f'{catalog}.{gold_schema}.staging_dim_{data_source}')
)

df_gold.show()

+-----------+--------------------+---------+--------------------+------+----------+-----------+
|customer_id|       customer_name|     city|            customer|market|  platform|    channel|
+-----------+--------------------+---------+--------------------+------+----------+-----------+
|     789622|Eliteathlete Nutr...|New Delhi|Eliteathlete Nutr...| Inida|Sports Bar|Acquisition|
|     789321|      Powersnack Hub|Hyderabad|Powersnack Hub - ...| Inida|Sports Bar|Acquisition|
|     789601|       Recovery Lane|Bengaluru|Recovery Lane - B...| Inida|Sports Bar|Acquisition|
|     789720|      Gameplan Foods|Bengaluru|Gameplan Foods - ...| Inida|Sports Bar|Acquisition|
|     789201|      Fitfuel Market|Bengaluru|Fitfuel Market - ...| Inida|Sports Bar|Acquisition|
|     789301|Athlete's Choice ...|Bengaluru|Athlete's Choice ...| Inida|Sports Bar|Acquisition|
|     789420|    Zenathlete Foods|Bengaluru|Zenathlete Foods ...| Inida|Sports Bar|Acquisition|
|     789202|      Fitfuel Market|Hydera

### Merge staging dimension with parent

In [0]:
display(spark.sql(f"""
MERGE INTO {catalog}.{gold_schema}.dim_{data_source} AS target
USING {catalog}.{gold_schema}.staging_dim_{data_source} AS source
ON 
    target.customer_code = source.customer_id

WHEN MATCHED THEN
UPDATE
SET
    market   = source.market,
    platform = source.platform,
    channel  = source.channel

WHEN NOT MATCHED THEN
INSERT (
    customer_code,
    customer,
    market,
    platform,
    channel)

VALUES (
    source.customer_id,
    source.customer,
    source.market,
    source.platform,
    source.channel
);
"""))

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
35,0,0,35
